# 🏆 Churn Prediction — Part 3: Hyperparameter Tuning with Optuna
## Squeeze maximum performance from each model

**What is Optuna?**  
Bayesian optimization library. Instead of trying ALL parameter combinations (GridSearch), it **intelligently picks** the next set of parameters based on what worked before. 10x faster than grid search.

**Strategy:** Tune on 3-fold CV (faster), then retrain best params on 5-fold for final OOF/test predictions.

**Input:** `X_train.csv`, `X_test.csv`, `y_train.csv`  
**Output:** `part3_oof_*.npy`, `part3_test_*.npy` (tuned predictions)


In [1]:
import pandas as pd, numpy as np, time, warnings, optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)


X_train = pd.read_csv('/Users/parveenkumarsharma/Downloads/playground-series-s6e3/X_train.csv')
X_test = pd.read_csv('/Users/parveenkumarsharma/Downloads/playground-series-s6e3/X_test.csv')
y_train = pd.read_csv('/Users/parveenkumarsharma/Downloads/playground-series-s6e3/y_train.csv').squeeze()
test_ids = pd.read_csv('/Users/parveenkumarsharma/Downloads/playground-series-s6e3/test_ids.csv').squeeze()

SPW = (y_train == 0).sum() / (y_train == 1).sum()
SEED = 42

# 3-fold for tuning (faster), 5-fold for final training
skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(f"Data: {X_train.shape[0]:,} × {X_train.shape[1]}")
print(f"Tuning CV: 3-fold | Final CV: 5-fold")


Data: 594,194 × 71
Tuning CV: 3-fold | Final CV: 5-fold


In [2]:
# Quick CV scorer (3-fold, for Optuna speed)
def quick_cv_score(train_fn, X, y, skf):
    scores = []
    for tr_idx, va_idx in skf.split(X, y):
        model = train_fn(X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[va_idx], y.iloc[va_idx])
        pred = model.predict_proba(X.iloc[va_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[va_idx], pred))
    return np.mean(scores)

# Full CV with OOF predictions (5-fold, for final predictions)
def full_cv(name, train_fn, X, y, X_test, skf, n_folds=5):
    oof = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    scores = []
    t0 = time.time()
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
        model = train_fn(X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[va_idx], y.iloc[va_idx])
        oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / n_folds
        fold_auc = roc_auc_score(y.iloc[va_idx], oof[va_idx])
        scores.append(fold_auc)
        print(f"  Fold {fold+1}: {fold_auc:.6f}")
    mean_auc = np.mean(scores)
    print(f"  ✅ {name}: {mean_auc:.6f} ± {np.std(scores):.6f} ({time.time()-t0:.1f}s)")
    return oof, test_preds, mean_auc


## Tune XGBoost with Optuna
Optuna will try ~50 different parameter combinations and find the best one.


In [3]:
# ============================================================
# TUNE XGBOOST (50 trials, 3-fold CV)
# ============================================================
def xgb_objective(trial):
    params = {
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.95),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'scale_pos_weight': SPW,
        'eval_metric': 'auc', 'tree_method': 'hist',
        'early_stopping_rounds': 100,
        'random_state': SEED, 'n_jobs': -1, 'verbosity': 0
    }
    def train_fn(X_tr, y_tr, X_va, y_va):
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        return m
    return quick_cv_score(train_fn, X_train, y_train, skf_tune)

print("▶ Tuning XGBoost (50 trials)...")
study_xgb = optuna.create_study(direction='maximize', study_name='xgb')
study_xgb.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f"\n  Best AUC (3-fold): {study_xgb.best_value:.6f}")
print(f"  Best params: {study_xgb.best_params}")


▶ Tuning XGBoost (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


  Best AUC (3-fold): 0.916378
  Best params: {'learning_rate': 0.033919507440902025, 'max_depth': 4, 'min_child_weight': 15, 'subsample': 0.9122949826813532, 'colsample_bytree': 0.5981087895038728, 'reg_alpha': 9.558188177456303, 'reg_lambda': 0.06037219229655164, 'gamma': 0.41009812789750927}


In [4]:
# Retrain XGBoost with best params (5-fold CV for final OOF)
best_xgb_params = study_xgb.best_params
def train_tuned_xgb(X_tr, y_tr, X_va, y_va):
    m = xgb.XGBClassifier(
        n_estimators=2000, early_stopping_rounds=100,
        scale_pos_weight=SPW, eval_metric='auc', tree_method='hist',
        random_state=SEED, n_jobs=-1, verbosity=0,
        **best_xgb_params
    )
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return m

print("▶ Retrain tuned XGBoost (5-fold):")
oof_xgb_t, test_xgb_t, score_xgb_t = full_cv("XGB-tuned", train_tuned_xgb, X_train, y_train, X_test, skf_final)


▶ Retrain tuned XGBoost (5-fold):
  Fold 1: 0.916102
  Fold 2: 0.917117
  Fold 3: 0.916650
  Fold 4: 0.917761
  Fold 5: 0.915007
  ✅ XGB-tuned: 0.916528 ± 0.000936 (477.4s)


## Tune LightGBM with Optuna

In [5]:
# ============================================================
# TUNE LIGHTGBM (50 trials)
# ============================================================
def lgb_objective(trial):
    params = {
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 96),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.95),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0, 1),
        'scale_pos_weight': SPW, 'subsample_freq': 1,
        'random_state': SEED, 'n_jobs': -1, 'verbose': -1
    }
    def train_fn(X_tr, y_tr, X_va, y_va):
        m = lgb.LGBMClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        return m
    return quick_cv_score(train_fn, X_train, y_train, skf_tune)

print("▶ Tuning LightGBM (50 trials)...")
study_lgb = optuna.create_study(direction='maximize', study_name='lgb')
study_lgb.optimize(lgb_objective, n_trials=50, show_progress_bar=True)

print(f"\n  Best AUC (3-fold): {study_lgb.best_value:.6f}")
print(f"  Best params: {study_lgb.best_params}")


▶ Tuning LightGBM (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[122]	valid_0's binary_logloss: 0.361293
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[121]	valid_0's binary_logloss: 0.359563
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[122]	valid_0's binary_logloss: 0.360367
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[29]	valid_0's binary_logloss: 0.365104
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[30]	valid_0's binary_logloss: 0.363248
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[30]	valid_0's binary_logloss: 0.363986
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[24]	valid_0's binary_logloss: 0.362322
Training until validation scores don't

In [6]:
# Retrain LightGBM with best params (5-fold)
best_lgb_params = study_lgb.best_params
def train_tuned_lgb(X_tr, y_tr, X_va, y_va):
    m = lgb.LGBMClassifier(
        n_estimators=2000, scale_pos_weight=SPW, subsample_freq=1,
        random_state=SEED, n_jobs=-1, verbose=-1, **best_lgb_params
    )
    m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    return m

print("▶ Retrain tuned LightGBM (5-fold):")
oof_lgb_t, test_lgb_t, score_lgb_t = full_cv("LGB-tuned", train_tuned_lgb, X_train, y_train, X_test, skf_final)


▶ Retrain tuned LightGBM (5-fold):
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[144]	valid_0's binary_logloss: 0.36007
  Fold 1: 0.913081
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[142]	valid_0's binary_logloss: 0.361043
  Fold 2: 0.914036
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[141]	valid_0's binary_logloss: 0.359631
  Fold 3: 0.913557
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[146]	valid_0's binary_logloss: 0.357297
  Fold 4: 0.914717
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[141]	valid_0's binary_logloss: 0.361684
  Fold 5: 0.912052
  ✅ LGB-tuned: 0.913488 ± 0.000899 (77.3s)


## Tune CatBoost with Optuna

In [11]:
# ============================================================
# TUNE CATBOOST (30 trials — slower model, fewer trials)
# ============================================================
def cb_objective(trial):
    params = {
        'iterations': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
        'auto_class_weights': 'Balanced',
        'eval_metric': 'AUC', 'early_stopping_rounds': 100,
        'random_seed': SEED, 'verbose': 0
    }
    def train_fn(X_tr, y_tr, X_va, y_va):
        m = CatBoostClassifier(**params)
        m.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
        return m
    return quick_cv_score(train_fn, X_train, y_train, skf_tune)

print("▶ Tuning CatBoost (30 trials)...")
study_cb = optuna.create_study(direction='maximize', study_name='cb')
study_cb.optimize(cb_objective, n_trials=30, show_progress_bar=True)

print(f"\n  Best AUC (3-fold): {study_cb.best_value:.6f}")
print(f"  Best params: {study_cb.best_params}")





▶ Tuning CatBoost (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]


  Best AUC (3-fold): 0.916149
  Best params: {'learning_rate': 0.08363645321096681, 'depth': 5, 'l2_leaf_reg': 7.646539500167732, 'subsample': 0.8041606743254643, 'min_data_in_leaf': 28}


In [12]:
# Retrain CatBoost with best params (5-fold)
best_cb_params = study_cb.best_params
def train_tuned_cb(X_tr, y_tr, X_va, y_va):
    m = CatBoostClassifier(
        iterations=2000, auto_class_weights='Balanced', eval_metric='AUC',
        early_stopping_rounds=100, random_seed=SEED, verbose=0, **best_cb_params
    )
    m.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=0)
    return m

print("▶ Retrain tuned CatBoost (5-fold):")
oof_cb_t, test_cb_t, score_cb_t = full_cv("CB-tuned", train_tuned_cb, X_train, y_train, X_test, skf_final)


▶ Retrain tuned CatBoost (5-fold):
  Fold 1: 0.915934
  Fold 2: 0.916990
  Fold 3: 0.916483
  Fold 4: 0.917442
  Fold 5: 0.914827
  ✅ CB-tuned: 0.916335 ± 0.000907 (565.0s)


In [13]:
# ============================================================
# SAVE TUNED PREDICTIONS + SUBMISSION
# ============================================================
np.save('part3_oof_xgb.npy', oof_xgb_t)
np.save('part3_oof_lgb.npy', oof_lgb_t)
np.save('part3_oof_cb.npy', oof_cb_t)
np.save('part3_test_xgb.npy', test_xgb_t)
np.save('part3_test_lgb.npy', test_lgb_t)
np.save('part3_test_cb.npy', test_cb_t)

# Tuned boost average
oof_tuned_avg = (oof_xgb_t + oof_lgb_t + oof_cb_t) / 3
test_tuned_avg = (test_xgb_t + test_lgb_t + test_cb_t) / 3
score_tuned_avg = roc_auc_score(y_train, oof_tuned_avg)

sub = pd.DataFrame({'id': test_ids, 'Churn': test_tuned_avg})
sub.to_csv('submission_part3_tuned_avg.csv', index=False)

print("\n" + "=" * 55)
print("  PART 3 RESULTS (TUNED)")
print("=" * 55)
print(f"  XGBoost (tuned):      {score_xgb_t:.6f}")
print(f"  LightGBM (tuned):     {score_lgb_t:.6f}")
print(f"  CatBoost (tuned):     {score_cb_t:.6f}")
print(f"  Tuned Boost Average:  {score_tuned_avg:.6f}")
print(f"\n  ✅ Saved all OOF/test predictions + submission")



  PART 3 RESULTS (TUNED)
  XGBoost (tuned):      0.916528
  LightGBM (tuned):     0.913488
  CatBoost (tuned):     0.916335
  Tuned Boost Average:  0.916279

  ✅ Saved all OOF/test predictions + submission
